# T3 · Additivity Audit
Subtasks: (1) classify ten measures, (2) the ratio proof, (3) three questions. All in this one notebook.

## 1 · Classification

In [1]:
import pandas as pd
pd.set_option("display.max_colwidth", None, "display.width", 250)

rows = [
 (1,  "WeightKG",                 "Additive",      "Yes", "WeightKG as is",
      "Summable across every dimension, including time."),
 (2,  "FreightCostPKR",           "Additive",      "Yes", "FreightCostPKR as is",
      "Currency amount; summable everywhere."),
 (3,  "ShipmentCount (=1)",       "Additive",      "Yes", "ShipmentCount = 1 on every row",
      "Counting rows is a SUM of 1s; gives the leg count at any level."),
 (4,  "TrucksAvailable at a depot", "Semi-additive", "No",  "Daily TrucksAvailable in a periodic snapshot; report average, last, min or max over time",
      "A level at a moment: SUM across depots is valid, SUM across days is not (10 trucks for 30 days is not 300 trucks)."),
 (5,  "WarehouseStockKG",         "Semi-additive", "No",  "Daily closing stock in a periodic snapshot; report last or average over time",
      "Same as 4: add across warehouses or cargo classes, never across dates."),
 (6,  "CostPerKG",                "Non-additive",  "No",  "FreightCostPKR + WeightKG; compute DIVIDE(SUM, SUM)",
      "A ratio is correct only at the level it was computed at."),
 (7,  "OnTimePercent",            "Non-additive",  "No",  "OnTimeLegCount + DeliveredLegCount; compute DIVIDE(SUM, SUM)",
      "Percentages cannot be added or averaged without their weights."),
 (8,  "TruckUtilisationRate",     "Non-additive",  "No",  "KG carried (or tonne-km) + KG capacity (or capacity tonne-km); compute DIVIDE(SUM, SUM)",
      "Utilisation of a half-empty and a full truck is not the mean of the two rates unless capacities are equal."),
 (9,  "Days_Late",                "Additive",      "Yes", "Days_Late + DeliveredLegCount (denominator for averages)",
      "Total days of lateness sums correctly; NULL, not 0, while a leg is In Transit. Average days late is a ratio."),
 (10, "AverageWeightPerShipment", "Non-additive",  "No",  "Nothing: delete the column (WeightKG and ShipmentCount already exist)",
      "Average = SUM(WeightKG) / SUM(ShipmentCount), derived at query time."),
]
t3 = pd.DataFrame(rows, columns=["#", "Candidate measure", "Additive / Semi / Non", "Safe to SUM across time?", "What to store instead", "Notes"])
assert len(t3) == 10 and t3["Additive / Semi / Non"].isin(["Additive", "Semi-additive", "Non-additive"]).all()
t3

,#,Candidate measure,Additive / Semi / Non,Safe to SUM across time?,What to store instead,Notes
0,1,WeightKG,Additive,Yes,WeightKG as is,"Summable across every dimension, including time."
1,2,FreightCostPKR,Additive,Yes,FreightCostPKR as is,Currency amount; summable everywhere.
2,3,ShipmentCount (=1),Additive,Yes,ShipmentCount = 1 on every row,Counting rows is a SUM of 1s; gives the leg count at any level.
3,4,TrucksAvailable at a depot,Semi-additive,No,"Daily TrucksAvailable in a periodic snapshot; report average, last, min or max over time","A level at a moment: SUM across depots is valid, SUM across days is not (10 trucks for 30 days is not 300 trucks)."
4,5,WarehouseStockKG,Semi-additive,No,Daily closing stock in a periodic snapshot; report last or average over time,"Same as 4: add across warehouses or cargo classes, never across dates."
5,6,CostPerKG,Non-additive,No,"FreightCostPKR + WeightKG; compute DIVIDE(SUM, SUM)",A ratio is correct only at the level it was computed at.
6,7,OnTimePercent,Non-additive,No,"OnTimeLegCount + DeliveredLegCount; compute DIVIDE(SUM, SUM)",Percentages cannot be added or averaged without their weights.
7,8,TruckUtilisationRate,Non-additive,No,"KG carried (or tonne-km) + KG capacity (or capacity tonne-km); compute DIVIDE(SUM, SUM)",Utilisation of a half-empty and a full truck is not the mean of the two rates unless capacities are equal.
8,9,Days_Late,Additive,Yes,Days_Late + DeliveredLegCount (denominator for averages),"Total days of lateness sums correctly; NULL, not 0, while a leg is In Transit. Average days late is a ratio."
9,10,AverageWeightPerShipment,Non-additive,No,Nothing: delete the column (WeightKG and ShipmentCount already exist),"Average = SUM(WeightKG) / SUM(ShipmentCount), derived at query time."


## 2 · The proof: average of a ratio vs ratio of two sums (same 34 rows)

In [2]:
import os
p = next(x for x in ["Shipments_Unified.csv", "../Shipments_Unified.csv", "/mnt/user-data/uploads/Shipments_Unified.csv"] if os.path.exists(x))
src = pd.read_csv(p)

wrong = (src.CostPKR / src.WeightKG).mean()          # AVERAGE of per-row Cost/Weight
right = src.CostPKR.sum() / src.WeightKG.sum()       # DIVIDE(SUM(Cost), SUM(Weight))
gap   = right - wrong

print(f"WRONG  average of per-row ratio : {wrong:.4f} PKR/kg")
print(f"RIGHT  ratio of two sums        : {right:.4f} PKR/kg")
print(f"Gap                             : {gap:.4f} PKR/kg  ({gap / right:.2%} of the right figure)")
assert round(wrong, 4) == 8.0604 and round(right, 4) == 8.3090

WRONG  average of per-row ratio : 8.0604 PKR/kg
RIGHT  ratio of two sums        : 8.3090 PKR/kg
Gap                             : 0.2486 PKR/kg  (2.99% of the right figure)


In [3]:
# Roll-up test: ratios by mode (SourceSystem). Sums roll up to the national figure; averaged ratios do not.
by = src.groupby("SourceSystem").agg(cost=("CostPKR", "sum"), kg=("WeightKG", "sum"))
by["cost_per_kg"] = by.cost / by.kg
print(by.round(4))
print("Mean of mode ratios (wrong)      :", round(by.cost_per_kg.mean(), 4))
print("SUM(cost) / SUM(kg) from modes   :", round(by.cost.sum() / by.kg.sum(), 4))
assert round(by.cost.sum() / by.kg.sum(), 4) == round(right, 4)

                 cost      kg  cost_per_kg
SourceSystem                              
Marine        1618000  206500       7.8354
Rail          2585000  295250       8.7553
Road           605000   76900       7.8674
Mean of mode ratios (wrong)      : 8.1527
SUM(cost) / SUM(kg) from modes   : 8.309


## 3 · The three questions

**Q1. Which number goes on a board slide, and what exactly does the other one measure?**
**8.3090 PKR/kg**, the ratio of the two sums: total freight cost divided by total kilograms moved, i.e. what the average kilogram actually cost. The other, **8.0604**, is the unweighted mean of 34 per-leg rates: the rate of the typical leg, where a 5,500 kg road leg counts exactly as much as a 33,000 kg rail leg. It is a real quantity (typical price per leg), but it answers "what does a leg usually cost per kg", not "what did a kilogram cost", so it understates the figure the board is asking for by about 3%.

**Q2. Rule for "average cost per kg by province" so provincial numbers roll up to the national one.**
Store only `FreightCostPKR` and `WeightKG` at leg grain; every rate, at any province, mode or month, is `DIVIDE(SUM(cost), SUM(weight))` evaluated in that filter context. Never average the provincial rates, and never store them. Provincial numerators and denominators then add to the national ones, so the national rate is the ratio of the national sums.

**Q3. The one measure whose correct answer is "delete the column", and why a stored average can never be fixed later.**
Row **10, AverageWeightPerShipment**: its ingredients (`WeightKG`, `ShipmentCount`) are already rows 1 and 3, so nothing needs storing. (`CostPerKG` is derivable the same way, but row 10 is the stored average the question describes.) A stored average has lost its denominator: from "6,500" alone you cannot tell whether it came from 1 shipment or 100, so it cannot be re-aggregated to a new level or corrected when rows are added or restated. Only the numerator and denominator can be summed again.